# MM-Net — headline reproduction and seed stability

Two experiments, both trained live in this notebook.

**1. Reproduction.** Retrain the headline configuration (concatenative fusion,
ten-fold patient-independent, seed 42) and check it against the published numbers.
This also regenerates the per-fold results cache, which had drifted out of step
with the reported run.

**2. Seed stability.** The paper reports one seed. A reviewer is entitled to ask
whether the result is a property of the model or of the initialisation, so the same
configuration is retrained under five seeds and the spread across seeds is compared
with the spread across folds.

All code is the locked configuration extracted verbatim from
`1_MM_Net_reproduction.ipynb` into `MMNet_research/model/mmnet_core.py`, so nothing
here is a re-implementation.

In [1]:
import json
import os
import sys
import time

import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))

import mmnet_core as C  # noqa: E402  (loads data and defines the locked model)

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)
print("repo :", REPO)
print("device:", C.DEV)

cwd: D:\sleep-staging-psg\MMNet_research\MMNet_Submission\all_codes\notebooks | device: cuda | NVIDIA GeForce RTX 2060


subjects: 96 (SN28 dropped) | epochs: 89,532
stage %: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}
respiratory-event prevalence: 16.0%
parameters (concat): 773,254
training utilities defined.
repo : D:\sleep-staging-psg
device: cuda


## 1. Reproduce the headline configuration

Ten folds, concatenative fusion, seed 42 — the configuration the paper reports.

In [2]:
t0 = time.time()
HEAD = C.run_10fold(fusion="concat", keep=True, seed=42)
print("\ntrained 10 folds in %.1f min" % ((time.time() - t0) / 60))

PUB = {"acc": 0.7227, "mf1": 0.6510, "kappa": 0.6106, "auc": 0.7111, "ap": 0.3367}
print("\n%-10s %18s %10s %10s" % ("metric", "this run (mean+-sd)", "published", "diff"))
print("-" * 52)
for k in ("acc", "mf1", "kappa", "auc", "ap"):
    m, s = HEAD[k]
    print("%-10s %10.4f +-%.3f %10.4f %10.4f" % (k, m, s, PUB[k], m - PUB[k]))


trained 10 folds in 6.2 min

metric     this run (mean+-sd)  published       diff
----------------------------------------------------
acc            0.7227 +-0.039     0.7227    -0.0000
mf1            0.6510 +-0.038     0.6510    -0.0000
kappa          0.6106 +-0.053     0.6106    -0.0000
auc            0.7111 +-0.034     0.7111     0.0000
ap             0.3367 +-0.083     0.3367    -0.0000


In [3]:
# regenerate the per-fold cache so the released artifact matches this run
import csv

csv_path = os.path.join(OUT, "headline_concat_per_fold.csv")
with open(csv_path, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["fold", "acc", "mf1", "kappa", "apnea_auc", "apnea_ap"])
    for i, f in enumerate(HEAD["per_fold"]):
        w.writerow([i, f["acc"], f["mf1"], f["kappa"], f["auc"], f["ap"]])

json.dump({k: HEAD[k] for k in ("acc", "mf1", "kappa", "auc", "ap", "pcf", "per_fold")},
          open(os.path.join(OUT, "headline_concat.json"), "w"), indent=1)

print("wrote", csv_path)
for i, f in enumerate(HEAD["per_fold"]):
    print("  fold %d  acc %.4f  mF1 %.4f  kappa %.4f  AUC %.4f"
          % (i, f["acc"], f["mf1"], f["kappa"], f["auc"]))

wrote D:\sleep-staging-psg\MMNet_research\results\revision\runs\headline_concat_per_fold.csv
  fold 0  acc 0.7330  mF1 0.6604  kappa 0.6248  AUC 0.7629
  fold 1  acc 0.7291  mF1 0.6673  kappa 0.6251  AUC 0.7176
  fold 2  acc 0.7238  mF1 0.6380  kappa 0.6080  AUC 0.6475
  fold 3  acc 0.7707  mF1 0.6695  kappa 0.6679  AUC 0.7002
  fold 4  acc 0.7662  mF1 0.6918  kappa 0.6634  AUC 0.6735
  fold 5  acc 0.6389  mF1 0.5763  kappa 0.4924  AUC 0.7316
  fold 6  acc 0.7415  mF1 0.6976  kappa 0.6528  AUC 0.6886
  fold 7  acc 0.6746  mF1 0.5920  kappa 0.5496  AUC 0.7106
  fold 8  acc 0.7465  mF1 0.6736  kappa 0.6410  AUC 0.7225
  fold 9  acc 0.7028  mF1 0.6434  kappa 0.5807  AUC 0.7563


## 2. Seed stability

The same ten folds and the same configuration, retrained under five seeds. If the
spread across seeds is small relative to the spread across folds, the reported result
reflects the model and the cohort rather than a fortunate initialisation.

In [4]:
SEEDS = [42, 1, 7, 123, 2026]
runs = {42: HEAD}                      # seed 42 already trained above

for sd in SEEDS:
    if sd in runs:
        continue
    t = time.time()
    runs[sd] = C.run_10fold(fusion="concat", seed=sd)
    print("seed %-5d done in %.1f min   acc %.4f  AUC %.4f"
          % (sd, (time.time() - t) / 60, runs[sd]["acc"][0], runs[sd]["auc"][0]))

seed 1     done in 6.0 min   acc 0.7280  AUC 0.7027


seed 7     done in 5.4 min   acc 0.7208  AUC 0.7013


seed 123   done in 5.0 min   acc 0.7282  AUC 0.7067


seed 2026  done in 5.6 min   acc 0.7261  AUC 0.7014


In [5]:
print("%-8s %10s %10s %10s %10s" % ("seed", "acc", "macro-F1", "kappa", "resp AUC"))
print("-" * 52)
for sd in SEEDS:
    r = runs[sd]
    print("%-8d %10.4f %10.4f %10.4f %10.4f"
          % (sd, r["acc"][0], r["mf1"][0], r["kappa"][0], r["auc"][0]))

print("-" * 52)
for k, label in (("acc", "accuracy"), ("auc", "resp AUC")):
    across_seeds = np.std([runs[s][k][0] for s in SEEDS])
    across_folds = np.mean([runs[s][k][1] for s in SEEDS])
    print("%-10s  sd across seeds %.4f   mean sd across folds %.4f   ratio %.2f"
          % (label, across_seeds, across_folds, across_seeds / across_folds))

json.dump({str(s): {k: runs[s][k] for k in ("acc", "mf1", "kappa", "auc", "ap")}
           for s in SEEDS},
          open(os.path.join(OUT, "seed_stability.json"), "w"), indent=1)
print("\nwrote seed_stability.json")

seed            acc   macro-F1      kappa   resp AUC
----------------------------------------------------
42           0.7227     0.6510     0.6106     0.7111
1            0.7280     0.6498     0.6177     0.7027
7            0.7208     0.6389     0.6087     0.7013
123          0.7282     0.6506     0.6178     0.7067
2026         0.7261     0.6505     0.6160     0.7014
----------------------------------------------------
accuracy    sd across seeds 0.0029   mean sd across folds 0.0319   ratio 0.09
resp AUC    sd across seeds 0.0038   mean sd across folds 0.0347   ratio 0.11

wrote seed_stability.json


## Reading the result

The quantity that matters is the ratio in the last block. Variation between seeds that
is small relative to variation between folds means the uncertainty in the reported
number is dominated by which patients land in which fold — a property of a 96-patient
cohort — and not by the optimiser's starting point. That is the claim the paper needs,
and it is the honest way to report a single-seed headline.